# TT indicators for the 3 new mule accounts

**Purpose:** find telegraphic-transfer (TT) red flags that would have identified the 3 accounts business flagged, in a form
another monitoring system can use. Standalone and exploratory — not part of the behavioural screen.

TTs are identified by `TRAN_RMKS`: **IT…** = inward TT (money in), **OT…** = outward TT (money out). Bank charges
(`RPT_CODE` CHG/INT) are excluded.

**Steps**
1. Pull TTs: the full TT history of every mule account, plus all TTs on all USD accounts since `TT_FROM` (any account age) —
   the population another system's rule would run on.
2. Look at the 3 accounts' TTs as they are: remarks, narrations, amounts, timing, who sent and who received.
3. Measure TT features per account and see where the 3 sit against every other TT-using account.
4. Test candidate rules across a range of thresholds: mules caught vs alerts per month.
5. Check whether mules send to, or receive from, the same counterparties.

Read-only. Pulls are cached per month; Finacle is only queried for months not yet cached (and the current month).

In [1]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SYNTHETIC = False

NEW_MULES = ["UG6302468", "UG6300154", "UG6334629"]  # the 3 accounts (ACID or account number)
INCLUDE_WATCHLIST = True                              # the original mules, for checking rules on accounts not used to set them
TT_FROM = "2025-01-01"                                # book-wide TT window for comparison and alert volumes

AS_OF = pd.Timestamp.today().normalize()
OUT = Path("outputs/tt_indicators"); OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path("cache/tt_indicators" + ("_synthetic" if SYNTHETIC else "")); CACHE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 70)

## 1. Connections and queries

In [2]:
ENV_FILE = ".env"
try:
    from dotenv import load_dotenv
    load_dotenv(ENV_FILE, override=False)
except ImportError:
    print("python-dotenv not installed; relying on variables already in the environment")

def _engine(prefix):
    '''Copied from aml_screening.py. READ = Finacle, WRITE = reporting DB (SELECTs only here).'''
    from sqlalchemy import create_engine, URL
    full_url = os.environ.get(f"AML_{prefix}_URL")
    if full_url:
        return create_engine(full_url, fast_executemany=True)
    def req(name):
        val = os.environ.get(f"AML_{prefix}_{name}")
        if not val:
            raise RuntimeError(f"Set AML_{prefix}_{name} (or provide a full AML_{prefix}_URL)")
        return val
    url = URL.create("mssql+pyodbc", username=req("USER"), password=req("PWD"), host=req("HOST"),
                     port=int(os.environ.get(f"AML_{prefix}_PORT", "1433")), database=req("DB"),
                     query={"driver": os.environ.get("AML_ODBC_DRIVER", "ODBC Driver 17 for SQL Server")})
    return create_engine(url, fast_executemany=True)

_fin = _rep = None
def fin():
    global _fin, _rep
    if _fin is None:
        _fin, _rep = _engine("READ"), _engine("WRITE")
    return _fin
def rep():
    fin(); return _rep

HTD = "RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE"
GAM = "RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE"
COLS = '''hd.ACID, gm.FORACID, gm.ACCT_NAME, gm.ACCT_OPN_DATE, hd.TRAN_ID, hd.TRAN_DATE, hd.RCRE_TIME, hd.TRAN_AMT,
          hd.PART_TRAN_TYPE, hd.TRAN_RMKS, hd.TRAN_PARTICULAR, hd.RPT_CODE'''

# All TTs on USD accounts in one calendar month (any account age)
MONTH_TT_SQL = f'''
SELECT {COLS}
FROM {HTD} hd JOIN {GAM} gm ON hd.ACID = gm.ACID
WHERE gm.ACCT_CRNCY_CODE = 'USD'
  AND hd.TRAN_DATE >= '{{start}}' AND hd.TRAN_DATE < '{{end}}'
  AND (hd.TRAN_RMKS LIKE 'IT%' OR hd.TRAN_RMKS LIKE 'OT%')
'''
# Every transaction of the mule accounts, all time (to see TTs in the context of everything else)
MULE_TXN_SQL = f'''
SELECT {COLS}
FROM {HTD} hd JOIN {GAM} gm ON hd.ACID = gm.ACID
WHERE hd.ACID IN ({{ids}}) OR gm.FORACID IN ({{ids}})
'''

def _in_list(ids):
    ids = [str(i).strip() for i in ids]
    bad = [i for i in ids if not re.fullmatch(r"[A-Za-z0-9_\-]+", i)]
    if bad:
        raise ValueError(f"Unexpected identifier format: {bad[:5]}")
    return ",".join(f"'{i}'" for i in ids)

## 2. Synthetic TT data (dry-run only)

In [3]:
def make_synthetic(seed=21):
    '''Fake USD accounts with inward/outward TTs. Mules receive TTs from several remitters and send them on within hours to
    a few beneficiaries, some shared between mules; importers pay out regularly; exporters receive and hold. Not evidence.'''
    rng = np.random.default_rng(seed)
    rows, n_id = [], [0]
    names = [f"PARTY {i:04d} LTD" for i in range(600)]
    shared_benef = ["GLOBAL LINK TRADING FZE", "ORIENT STAR GENERAL TRADING"]
    def tt(acid, fora, name, opn, when, amt, kind, party):
        n_id[0] += 1
        rows.append(dict(ACID=acid, FORACID=fora, ACCT_NAME=name, ACCT_OPN_DATE=opn, TRAN_ID=f"T{n_id[0]}",
                         TRAN_DATE=when.normalize(), RCRE_TIME=when, TRAN_AMT=round(float(amt), 2),
                         PART_TRAN_TYPE="C" if kind == "IT" else "D", TRAN_RMKS=f"{kind}{rng.integers(10**6, 10**7)}",
                         TRAN_PARTICULAR=party, RPT_CODE=None))
    accts = []
    for i in range(1500):
        acid, fora = f"UG{i:07d}", f"9{i:09d}"
        opn = pd.Timestamp("2023-06-01") + pd.Timedelta(days=int(rng.integers(0, 1200)))
        start = max(opn, pd.Timestamp("2024-06-01"))
        if i < 13:                                                   # mules: fast in, fast out
            name = f"SYNTHETIC MULE {i:02d}"
            t0 = opn + pd.Timedelta(days=int(rng.integers(1, 40)), hours=9)
            for k in range(int(rng.integers(6, 15))):
                when = t0 + pd.Timedelta(days=int(rng.integers(0, 20)), hours=float(rng.uniform(0, 8)))
                amt = rng.uniform(40_000, 400_000)
                tt(acid, fora, name, opn, when, amt, "IT", f"IT/{rng.choice(names)}")
                tt(acid, fora, name, opn, when + pd.Timedelta(hours=float(rng.uniform(1, 30))), amt * rng.uniform(0.93, 0.99),
                   "OT", f"OT/{rng.choice(shared_benef) if rng.random() < .5 else rng.choice(names)}")
        else:
            name = f"SYNTHETIC CUSTOMER {i:05d}"; r = rng.random()
            for m in range(int(rng.integers(2, 18))):
                when = start + pd.Timedelta(days=int(rng.integers(0, max((AS_OF - start).days, 2))), hours=float(rng.uniform(8, 17)))
                if when > AS_OF:
                    continue
                if r < .45:      # importer: pays suppliers, funded by local deposits (not TT)
                    tt(acid, fora, name, opn, when, rng.uniform(5_000, 150_000), "OT", f"OT/{rng.choice(names)}")
                elif r < .85:    # exporter: receives, pays out occasionally, slowly
                    amt = rng.uniform(5_000, 200_000)
                    tt(acid, fora, name, opn, when, amt, "IT", f"IT/{rng.choice(names)}")
                    if rng.random() < .3:
                        tt(acid, fora, name, opn, when + pd.Timedelta(days=int(rng.integers(5, 40))), amt * .5, "OT",
                           f"OT/{rng.choice(names)}")
                else:            # fast legitimate pass-through (e.g. a trading intermediary)
                    amt = rng.uniform(20_000, 300_000)
                    tt(acid, fora, name, opn, when, amt, "IT", f"IT/{rng.choice(names)}")
                    tt(acid, fora, name, opn, when + pd.Timedelta(hours=float(rng.uniform(2, 60))), amt * .97, "OT",
                       f"OT/{rng.choice(names)}")
    df = pd.DataFrame(rows)
    df = df[df.RCRE_TIME <= AS_OF]
    return df, [f"UG{i:07d}" for i in range(3)], [f"UG{i:07d}" for i in range(3, 13)]

## 3. Load — cache first

In [4]:
if SYNTHETIC:
    syn_tt, syn_new, syn_watch = make_synthetic()
    NEW_MULES = syn_new
watch = (syn_watch if SYNTHETIC else
         (pd.read_sql("SELECT acid FROM aml_watchlist WHERE active = 1", rep())["acid"].tolist() if INCLUDE_WATCHLIST else []))
mule_ids = list(dict.fromkeys([str(x) for x in NEW_MULES] + [str(x) for x in watch]))
queries = []

# full history of the mule accounts
p = CACHE_DIR / f"mule_txns_{len(mule_ids)}.pkl"
if p.exists():
    mule_tx = pd.read_pickle(p)
else:
    mule_tx = (syn_tt[syn_tt.ACID.isin(mule_ids) | syn_tt.FORACID.isin(mule_ids)].copy() if SYNTHETIC
               else pd.read_sql(MULE_TXN_SQL.format(ids=_in_list(mule_ids)), fin()))
    mule_tx.to_pickle(p); queries.append(f"mule accounts: {len(mule_tx):,} transactions")

# book-wide TTs by month; past months are final, the current month is always refreshed
parts = []
for m in pd.period_range(pd.Timestamp(TT_FROM).to_period("M"), AS_OF.to_period("M"), freq="M"):
    f = CACHE_DIR / f"tt_{m}.pkl"
    if not f.exists() or m == AS_OF.to_period("M"):
        s, e = m.start_time, (m + 1).start_time
        t = (syn_tt[(syn_tt.TRAN_DATE >= s) & (syn_tt.TRAN_DATE < e)].copy() if SYNTHETIC
             else pd.read_sql(MONTH_TT_SQL.format(start=s.strftime("%Y-%m-%d"), end=e.strftime("%Y-%m-%d")), fin()))
        t.to_pickle(f); queries.append(f"TTs {m}: {len(t):,}")
    parts.append(pd.read_pickle(f))
book_tt = pd.concat(parts, ignore_index=True)
print("Finacle queries this run:", "; ".join(queries) if queries else "none — everything served from cache")

def prep(df):
    df = df.copy()
    for col in ("TRAN_DATE", "RCRE_TIME", "ACCT_OPN_DATE"):
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["when"] = df.RCRE_TIME.fillna(df.TRAN_DATE)
    df["rmk"] = df.TRAN_RMKS.fillna("").str.strip().str.upper()
    df["tt"] = np.where(df.rmk.str.startswith("IT"), "IT", np.where(df.rmk.str.startswith("OT"), "OT", ""))
    df["is_charge"] = df.RPT_CODE.fillna("").str.upper().isin(["CHG", "INT"])
    df["TRAN_AMT"] = df.TRAN_AMT.astype(float).abs()
    df["party"] = (df.TRAN_PARTICULAR.fillna("").str.upper().str.replace(r"^\s*(IT|OT)\s*[/:-]?\s*", "", regex=True)
                     .str.replace(r"\d{6,}", "#", regex=True).str.replace(r"\s+", " ", regex=True).str.strip())   # mask long reference numbers only
    return df

mule_tx, book_tt = prep(mule_tx), prep(book_tt)
book_tt = book_tt[(book_tt.tt != "") & ~book_tt.is_charge]
newset = set(map(str, NEW_MULES))
acct_ids = mule_tx[["ACID", "FORACID", "ACCT_NAME"]].drop_duplicates("ACID")
new_acids = acct_ids[acct_ids.ACID.astype(str).isin(newset) | acct_ids.FORACID.astype(str).isin(newset)].ACID.tolist()
old_acids = [a for a in acct_ids.ACID if a not in new_acids]
group = {**{a: "new mule" for a in new_acids}, **{a: "original mule" for a in old_acids}}
print(f"Mule accounts found: {len(new_acids)} new, {len(old_acids)} original | "
      f"book TTs since {TT_FROM}: {len(book_tt):,} on {book_tt.ACID.nunique():,} accounts")

Finacle queries this run: mule accounts: 8,892 transactions; TTs 2025-01: 98,471; TTs 2025-02: 96,367; TTs 2025-03: 102,558; TTs 2025-04: 100,665; TTs 2025-05: 104,308; TTs 2025-06: 101,421; TTs 2025-07: 99,717; TTs 2025-08: 88,585; TTs 2025-09: 95,252; TTs 2025-10: 96,380; TTs 2025-11: 86,067; TTs 2025-12: 104,771; TTs 2026-01: 83,878; TTs 2026-02: 88,418; TTs 2026-03: 97,944; TTs 2026-04: 94,756; TTs 2026-05: 90,841; TTs 2026-06: 97,707; TTs 2026-07: 102,344; TTs 2026-08: 98,807; TTs 2026-09: 99,961; TTs 2026-10: 91,258
Mule accounts found: 3 new, 10 original | book TTs since 2025-01-01: 1,653,645 on 17,802 accounts


## 4. The 3 accounts as they are

What their TTs look like before any measuring: the share of their money that moved by TT, sample remarks and narrations,
and the largest senders and receivers.

In [5]:
names = acct_ids.set_index("ACID").ACCT_NAME
for a in new_acids:
    t = mule_tx[(mule_tx.ACID == a) & ~mule_tx.is_charge]
    cr, dr = t[t.PART_TRAN_TYPE.str.upper() == "C"], t[t.PART_TRAN_TYPE.str.upper() == "D"]
    print("=" * 110)
    print(f"{names[a]}  |  opened {t.ACCT_OPN_DATE.min():%Y-%m-%d}  |  credits ${cr.TRAN_AMT.sum():,.0f}, of which inward TT "
          f"{cr[cr.tt == 'IT'].TRAN_AMT.sum() / max(cr.TRAN_AMT.sum(), 1):.0%}  |  debits ${dr.TRAN_AMT.sum():,.0f}, of which "
          f"outward TT {dr[dr.tt == 'OT'].TRAN_AMT.sum() / max(dr.TRAN_AMT.sum(), 1):.0%}")
    print("\nRemark prefixes (first 2 characters) by direction:")
    print(t.assign(prefix=t.rmk.str[:2], dir=t.PART_TRAN_TYPE.str.upper()).groupby(["dir", "prefix"]).TRAN_AMT.agg(["size", "sum"])
            .rename(columns={"size": "count", "sum": "$"}).round(0).to_string())
    tt_ = t[t.tt != ""].sort_values("when")
    print("\nFirst 15 TTs as recorded:")
    print(tt_[["when", "tt", "TRAN_AMT", "TRAN_RMKS", "TRAN_PARTICULAR"]].head(15).to_string(index=False))
    print("\nLargest counterparties by TT:")
    print(tt_.groupby(["tt", "party"]).TRAN_AMT.agg(["size", "sum"]).sort_values("sum", ascending=False).head(8)
             .rename(columns={"size": "count", "sum": "$"}).round(0).to_string())

TYR TECH -SMC LTD  |  opened 2026-02-26  |  credits $5,074,335, of which inward TT 100%  |  debits $5,072,053, of which outward TT 0%

Remark prefixes (first 2 characters) by direction:
            count          $
dir prefix                  
C               5       60.0
    07          1       41.0
    BO          1        5.0
    IT         62  5074200.0
    TA          1       29.0
D               7       18.0
    AC          1        2.0
    FE          1       10.0
    PA         64  5071995.0
    TA          1       29.0

First 15 TTs as recorded:
               when tt  TRAN_AMT        TRAN_RMKS                     TRAN_PARTICULAR
2026-03-03 08:00:37 IT    3492.5 IT26062UG0163522             SHIVINDER SINGH PUREWAL
2026-03-03 08:00:38 IT    6000.0 IT26062UG0163530                           THOA PHAM
2026-03-04 08:00:37 IT    9990.0 IT26063UG0164026                       Kristy Nguyen
2026-03-04 11:12:49 IT    4000.0 IT26063UG0164230                           THOA PHAM
2026-03-0

## 5. TT features per account, and where the 3 sit against the book

Measured per account on its TTs since `TT_FROM` (mules: their full TT history). `hours_to_out` follows each inward TT's
money, first in first out, to the outward TTs that took it out.

In [6]:
def clear_times(g):
    '''For each inward TT, hours until outward TTs (first in, first out) had taken out its full amount; NaN if never.'''
    g = g.sort_values("when")
    it, ot = g[g.tt == "IT"], g[g.tt == "OT"]
    if it.empty:
        return pd.Series(dtype=float)
    cum_out = ot.TRAN_AMT.cumsum().to_numpy(); t_out = ot.when.to_numpy()
    need = it.TRAN_AMT.cumsum().to_numpy()
    k = np.searchsorted(cum_out, need - 1e-6)              # first outward TT at which cumulative out covers this IT
    hrs = np.full(len(it), np.nan)
    ok = k < len(cum_out)
    hrs[ok] = (t_out[k[ok]] - it.when.to_numpy()[ok]) / np.timedelta64(1, "h")
    return pd.Series(np.clip(hrs, 0, None), index=it.index)

def it_clear_table(df):
    '''One row per inward TT with the hours its money took to leave (computed once, reused everywhere).'''
    parts = []
    for a, g in df.groupby("ACID"):
        h = clear_times(g)
        if len(h):
            parts.append(g.loc[h.index, ["ACID", "when", "TRAN_AMT"]].assign(hrs=h.to_numpy()))
    return pd.concat(parts) if parts else pd.DataFrame(columns=["ACID", "when", "TRAN_AMT", "hrs"])

def features(df, itc):
    rows = {}
    hrs_by = {a: g for a, g in itc.groupby("ACID")}
    for a, g in df.groupby("ACID"):
        g = g.sort_values("when")
        it, ot = g[g.tt == "IT"], g[g.tt == "OT"]
        ic = hrs_by.get(a, itc.iloc[:0]); h, w = ic.hrs, ic.TRAN_AMT
        f = {"IT count": len(it), "IT total $k": it.TRAN_AMT.sum() / 1e3,
             "OT count": len(ot), "OT total $k": ot.TRAN_AMT.sum() / 1e3,
             "OT / IT": ot.TRAN_AMT.sum() / it.TRAN_AMT.sum() if len(it) else np.nan,
             "IT money out within 24h": (w[h <= 24].sum() / w.sum()) if len(ic) else np.nan,
             "IT money out within 72h": (w[h <= 72].sum() / w.sum()) if len(ic) else np.nan,
             "median hours to out": h.median() if len(h) else np.nan,
             "distinct senders": it.party.nunique(), "distinct receivers": ot.party.nunique(),
             "largest IT $k": it.TRAN_AMT.max() / 1e3 if len(it) else 0,
             "round-thousand TTs": (g.TRAN_AMT % 1000 == 0).mean(),
             "days open at first IT": (it.when.min() - g.ACCT_OPN_DATE.min()).days if len(it) else np.nan,
             "max IT in 7 days $k": (it.dropna(subset=["when"]).sort_values("when").set_index("when").TRAN_AMT
                                     .rolling("7D").sum().max() / 1e3) if len(it) else 0,
             "active TT days": g.when.dt.normalize().nunique()}
        rows[a] = f
    return pd.DataFrame.from_dict(rows, orient="index")

mule_tt = mule_tx[(mule_tx.tt != "") & ~mule_tx.is_charge]          # mules: full TT history
other_tt = book_tt[~book_tt.ACID.isin(group)]                          # everyone else: TTs since TT_FROM
itc = it_clear_table(pd.concat([mule_tt, other_tt]))
mule_feat = features(mule_tt, itc)
book_feat = features(other_tt, itc)
book_it = book_feat[book_feat["IT count"] > 0]

def pct(col, v):
    s = book_it[col].dropna()
    return np.nan if pd.isna(v) or s.empty else round(100 * (s <= v).mean())

view = mule_feat.loc[[a for a in new_acids if a in mule_feat.index]].T
view.columns = [str(names[a])[:22] for a in view.columns]
for a, colname in zip([a for a in new_acids if a in mule_feat.index], list(view.columns)):
    view[colname + " pctile"] = [pct(c, mule_feat.loc[a, c]) for c in view.index]
view["original mules (median)"] = mule_feat.loc[[a for a in old_acids if a in mule_feat.index]].median()
view["book accounts with IT (median)"] = book_it.median()
view["book p95"] = book_it.quantile(.95)
print(f"Compared with {len(book_it):,} non-mule USD accounts that received an inward TT since {TT_FROM}. "
      "Percentile = share of those accounts at or below the mule's value.")
display(view.round(2))

Compared with 13,464 non-mule USD accounts that received an inward TT since 2025-01-01. Percentile = share of those accounts at or below the mule's value.


,TYR TECH -SMC LTD,THRIFTOPIA UGANDA LIMI,QUENSPIRE UGANDA LIMIT,TYR TECH -SMC LTD pctile,THRIFTOPIA UGANDA LIMI pctile,QUENSPIRE UGANDA LIMIT pctile,original mules (median),book accounts with IT (median),book p95
IT count,62.00,4.00,5.00,96.0,48.0,53.0,21.50,5.00,50.00
IT total $k,5074.20,2289.80,736.56,98.0,97.0,92.0,1674.24,19.47,1388.76
OT count,0.00,0.00,0.00,60.0,60.0,60.0,2.50,0.00,56.00
OT total $k,0.00,0.00,0.00,60.0,60.0,60.0,139.90,0.00,828.77
OT / IT,0.00,0.00,0.00,60.0,60.0,60.0,0.01,0.00,13.35
IT money out within 24h,0.00,0.00,0.00,84.0,84.0,84.0,0.00,0.00,1.00
IT money out within 72h,0.00,0.00,0.00,83.0,83.0,83.0,0.00,0.00,1.00
median hours to out,NaN,NaN,NaN,NaN,NaN,NaN,38.63,718.12,11226.40
distinct senders,36.00,1.00,4.00,98.0,41.0,83.0,17.00,2.00,12.00
distinct receivers,0.00,0.00,0.00,60.0,60.0,60.0,2.00,0.00,29.00


## 6. Candidate rules across thresholds

Each rule is something a TT-monitoring system can express. Every row shows, for one threshold setting: which of the 3 new
mules it catches, how many original mules, and how many other accounts it would have raised per month since `TT_FROM`.
Pick the row with the best balance — and prefer settings that also catch original mules, not only these 3.

In [7]:
months = max(len(pd.period_range(pd.Timestamp(TT_FROM).to_period("M"), AS_OF.to_period("M"))) - 1, 1)
all_feat = pd.concat([mule_feat, book_feat])
kind = pd.Series({a: group.get(a, "other") for a in all_feat.index})

# event-level pass-through: an inward TT of at least $X whose money had all left by outward TT within H hours
def passthrough_hits(X, H):
    return pd.Index(itc[(itc.TRAN_AMT >= X) & (itc.hrs <= H)].ACID.unique())

def score(hit_accounts, label):
    hit = pd.Index(hit_accounts)
    return {"rule": label,
            "new mules caught": ", ".join(str(names[a])[:18] for a in new_acids if a in hit) or "none",
            "of 3": sum(a in hit for a in new_acids),
            "original mules": f"{sum(a in hit for a in old_acids)} of {len(old_acids)}",
            "other accounts": int(sum(kind.get(a, 'other') == 'other' for a in hit)),
            "per month": round(sum(kind.get(a, 'other') == 'other' for a in hit) / months, 1)}

rows = []
for X in (25_000, 50_000, 100_000, 250_000):
    for H in (24, 48, 72):
        rows.append(score(passthrough_hits(X, H), f"T1 inward TT >= ${X/1e3:,.0f}k fully sent out by OT within {H}h"))
for X in (100_000, 250_000, 500_000, 1_000_000):
    for D in (30, 90):
        f = all_feat
        rows.append(score(f[(f["days open at first IT"] <= D) & (f["max IT in 7 days $k"] * 1e3 >= X)].index,
                          f"T2 account <= {D} days old receives >= ${X/1e3:,.0f}k by IT within 7 days"))
for r in (0.8, 0.9):
    for X in (250_000, 500_000, 1_000_000):
        f = all_feat
        rows.append(score(f[(f["OT / IT"] >= r) & (f["IT total $k"] * 1e3 >= X) & (f["IT money out within 72h"] >= 0.8)].index,
                          f"T3 IT total >= ${X/1e3:,.0f}k, OT/IT >= {r:.0%}, 80% out within 72h"))
grid = pd.DataFrame(rows).set_index("rule")
display(grid)
print(f"Alert counts are distinct accounts over {months} complete months since {TT_FROM} (all account ages). "
      "Mules are judged on their full TT history, so original mules active before TT_FROM still count.")

,new mules caught,of 3,original mules,other accounts,per month
rule,,,,,
T1 inward TT >= $25k fully sent out by OT within 24h,none,0,2 of 10,608,29.0
T1 inward TT >= $25k fully sent out by OT within 48h,none,0,3 of 10,617,29.4
T1 inward TT >= $25k fully sent out by OT within 72h,none,0,3 of 10,625,29.8
T1 inward TT >= $50k fully sent out by OT within 24h,none,0,1 of 10,431,20.5
T1 inward TT >= $50k fully sent out by OT within 48h,none,0,2 of 10,436,20.8
T1 inward TT >= $50k fully sent out by OT within 72h,none,0,2 of 10,446,21.2
T1 inward TT >= $100k fully sent out by OT within 24h,none,0,1 of 10,291,13.9
T1 inward TT >= $100k fully sent out by OT within 48h,none,0,1 of 10,296,14.1
T1 inward TT >= $100k fully sent out by OT within 72h,none,0,1 of 10,301,14.3


Alert counts are distinct accounts over 21 complete months since 2025-01-01 (all account ages). Mules are judged on their full TT history, so original mules active before TT_FROM still count.


## 7. Shared counterparties

Do mules send money to, or receive it from, the same parties — and how many other accounts deal with those parties?
A receiver used by two or more mules is a strong, simple watch-list rule for any system. Narration names are rough, so
check the names before using them.

In [8]:
mt = mule_tt[mule_tt.party.str.len() > 3]
shared = (mt.groupby(["tt", "party"]).agg(mules=("ACID", "nunique"), mule_names=("ACCT_NAME", lambda s: ", ".join(sorted(set(map(lambda x: str(x)[:16], s))))),
                                          mule_total_k=("TRAN_AMT", lambda s: round(s.sum() / 1e3)))
            .query("mules >= 2"))
use = other_tt.groupby(["tt", "party"]).ACID.nunique()
shared["other accounts using it"] = [int(use.get((t, p), 0)) for t, p in shared.index]
# most useful first: used by many mules and few other accounts (a generic narration is used by everyone)
shared = shared.sort_values(["mules", "other accounts using it", "mule_total_k"], ascending=[False, True, False])
print(f"Counterparties used by 2 or more mule accounts: {len(shared)}")
display(shared.head(30))

Counterparties used by 2 or more mule accounts: 68


mules                                                             mule_names  mule_total_k  other accounts using it
tt party                                                                                                                                                   
IT SANDY TRAN                               4  AETHERGRID TECHN, KWELIVEST INVEST, NEBULACORE INNOV, PACIFIC PRIN...           163                        4
   1/THOMAS JURGEN SCHUNN                   3                   QUENSPIRE UGANDA, THRIFTOPIA UGAND, TYR TECH -SMC LT          3339                        0
   GARY BRADSHAW /OR GILMER GRAYLL BR       3                   AETHERGRID TECHN, KWELIVEST INVEST, NEBULACORE INNOV           258                        1
   BILLY NGO                                3                   AETHERGRID TECHN, KWELIVEST INVEST, NEBULACORE INNOV            43                        1
   BRET BISHOP                              3                   AETHERGRID TECHN, NEBULACORE INNOV, PACIFIC PRINCE I           205                        2
   PHUONG M TRIEU                           3                   AETHERGRID TECHN, NEBULACORE INNOV, PACIFIC PRINCE I           135                        2
   KELLY TRAN                               3                   AETHERGRID TECHN, NEBULACORE INNOV, PACIFIC PRINCE I            33                        2
   CHI LUU LLC DBA LIVING COLORS NAILS      3                   KWELIVEST INVEST, NEBULACORE INNOV, PACIFIC PRINCE I            78                        4
   CHI V PHAM                               3                   AETHERGRID TECHN, NEBULACORE INNOV, PACIFIC PRINCE I            53                        4
   VINH DUC NGUYEN                          3                   KWELIVEST INVEST, NEBULACORE INNOV, PACIFIC PRINCE I            47                        4
   DUYKIM TRAN                              3                   KWELIVEST INVEST, NEBULACORE INNOV, PACIFIC PRINCE I            28                        4
   QUYEN PHAM                               3                   AETHERGRID TECHN, NEBULACORE INNOV, PACIFIC PRINCE I           124                        5
   KPC INTERNATIONAL CORPORATION            3                   AETHERGRID TECHN, PACIFIC PRINCE I, TYR TECH -SMC LT          1642                       14
OT PAYMENT FOR GOODS                        3                      EMIC VENTURES, HESHIMA PRIME LO, SHAUPAM ENTERPRI         13922                       97
IT CINDY HUYNH                              2                                     KWELIVEST INVEST, NEBULACORE INNOV           201                        0
   MICHAEL GUTHKELCH                        2                                     KWELIVEST INVEST, NEBULACORE INNOV           177                        0
   MAI THANH                                2                                     AETHERGRID TECHN, NEBULACORE INNOV            89                        0
   DUYEN TRUONG                             2                                     KWELIVEST INVEST, NEBULACORE INNOV            39                        0
   CHRISTINE THI MAI                        2                                     KWELIVEST INVEST, NEBULACORE INNOV            15                        0
   AMY WOLFE                                2                                     AETHERGRID TECHN, NEBULACORE INNOV           593                        1
   TRINH DUONG                              2                                     AETHERGRID TECHN, NEBULACORE INNOV           166                        1
   METE MUTLUOGLU                           2                                     KWELIVEST INVEST, NEBULACORE INNOV           165                        1
   ANITA TRINH                              2                                     KWELIVEST INVEST, NEBULACORE INNOV           145                        1
   CASSIDY NGUYEN CHI                       2                                     AETHERGRID TECHN, NEBULACORE INNOV           140          

## 8. Export

In [9]:
view.to_csv(OUT / "tt_profile_new_mules.csv")
grid.to_csv(OUT / "tt_rule_grid.csv")
shared.to_csv(OUT / "tt_shared_counterparties.csv")
mule_feat.assign(group=mule_feat.index.map(group)).to_csv(OUT / "tt_features_mules.csv")
print("Saved to", OUT.resolve())

Saved to /home/A270827/experiments/AML/outputs/tt_indicators
